In [ ]:
# Установка необходимых библиотек
!pip install transformers datasets torch scikit-learn matplotlib seaborn pandas numpy tqdm -q

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from transformers import DistilBertTokenizer, DistilBertModel
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import random

# Установка seed для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Проверка доступности GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

In [ ]:
# Загрузка датасета
print("Загрузка датасета SST-2...")
dataset = load_dataset("glue", "sst2")
train_dataset = dataset['train']
val_dataset = dataset['validation']

# Для ускорения эксперимента возьмем подвыборку
train_sample = train_dataset.select(range(2000))
val_sample = val_dataset.select(range(500))

print(f"Размер обучающей выборки: {len(train_sample)}")
print(f"Размер валидационной выборки: {len(val_sample)}")

# Просмотр примеров данных
print("\nПримеры из датасета:")
for i in range(3):
    print(f"Пример {i+1}:")
    print(f"  Текст: {train_sample[i]['sentence']}")
    print(f"  Метка: {train_sample[i]['label']} ({'Позитивный' if train_sample[i]['label'] == 1 else 'Негативный'})")
    print()

In [ ]:
# Загрузка модели и токенизатора
print("Загрузка модели DistilBERT...")
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
model = DistilBertModel.from_pretrained(model_name, output_hidden_states=True)
model.to(device)
model.eval()

print(f"Модель DistilBERT имеет {model.config.num_hidden_layers} скрытых слоев")
print(f"Размерность эмбеддингов: {model.config.dim}")
print(f"Количество attention heads: {model.config.n_heads}")

In [ ]:
# Функция для извлечения представлений из разных слоев
def extract_layer_representations(texts, labels, max_length=128):
    """
    Извлекает представления из всех слоев модели для заданных текстов
    """
    all_layer_representations = []
    all_labels = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), 32), desc="Извлечение представлений"):
            batch_texts = texts[i:i+32]
            batch_labels = labels[i:i+32]

            # Токенизация
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            # Прямой проход через модель
            outputs = model(**inputs)

            # Получаем все скрытые состояния
            # hidden_states содержит [embeddings, layer1, layer2, ..., layer6]
            hidden_states = outputs.hidden_states

            # Для каждого примера в батче
            for batch_idx in range(len(batch_texts)):
                example_representations = []

                # Для каждого слоя (включая эмбеддинги)
                for layer_idx, layer_state in enumerate(hidden_states):
                    # Берем [CLS] токен как представление всего предложения
                    cls_embedding = layer_state[batch_idx, 0, :].cpu().numpy()
                    example_representations.append(cls_embedding)

                all_layer_representations.append(example_representations)
                all_labels.append(batch_labels[batch_idx])

    # Преобразуем в массив: [примеры, слои, размерность]
    all_layer_representations = np.array(all_layer_representations)
    all_labels = np.array(all_labels)

    return all_layer_representations, all_labels

In [ ]:
# Подготовка данных
print("\nПодготовка данных для извлечения представлений...")
train_texts = train_sample['sentence']
train_labels = train_sample['label']

val_texts = val_sample['sentence']
val_labels = val_sample['label']

# Извлечение представлений
print("\nИзвлечение представлений для обучающей выборки...")
train_representations, train_labels_arr = extract_layer_representations(train_texts, train_labels)

print("\nИзвлечение представлений для валидационной выборки...")
val_representations, val_labels_arr = extract_layer_representations(val_texts, val_labels)

print(f"\nФорма представлений: {train_representations.shape}")
print(f"Количество слоев: {train_representations.shape[1]}")
print(f"Размерность каждого представления: {train_representations.shape[2]}")

# Проверка баланса классов
print(f"\nРаспределение классов в обучающей выборке:")
print(f"  Класс 0 (негативный): {np.sum(train_labels_arr == 0)} примеров")
print(f"  Класс 1 (позитивный): {np.sum(train_labels_arr == 1)} примеров")

In [ ]:
# Функция для оценки качества представлений разных слоев
def evaluate_layers_classification(train_reps, train_labels, val_reps, val_labels, classifier_type='logistic'):
    """
    Оценивает качество классификации на представлениях каждого слоя
    """
    num_layers = train_reps.shape[1]
    results = []

    for layer_idx in tqdm(range(num_layers), desc="Оценка слоев"):
        # Получаем представления для текущего слоя
        X_train = train_reps[:, layer_idx, :]
        X_val = val_reps[:, layer_idx, :]

        # Выбор классификатора
        if classifier_type == 'logistic':
            clf = LogisticRegression(max_iter=1000, random_state=42)
        elif classifier_type == 'svm':
            clf = SVC(kernel='linear', random_state=42)
        elif classifier_type == 'rf':
            clf = RandomForestClassifier(n_estimators=100, random_state=42)
        else:
            clf = LogisticRegression(max_iter=1000, random_state=42)

        # Обучение классификатора
        clf.fit(X_train, train_labels)

        # Предсказание
        y_pred = clf.predict(X_val)

        # Расчет метрик
        accuracy = accuracy_score(val_labels, y_pred)
        f1 = f1_score(val_labels, y_pred)
        precision = precision_score(val_labels, y_pred)
        recall = recall_score(val_labels, y_pred)

        results.append({
            'layer': layer_idx,
            'accuracy': accuracy,
            'f1_score': f1,
            'precision': precision,
            'recall': recall
        })

    return pd.DataFrame(results)

In [ ]:
# Оценка качества для каждого слоя
print("\nОценка качества классификации для каждого слоя...")
results_df = evaluate_layers_classification(
    train_representations, train_labels_arr,
    val_representations, val_labels_arr,
    classifier_type='logistic'
)

# Вывод первых результатов
print("\nРезультаты для первых 3 слоев:")
print(results_df.head(3).to_string(index=False))

In [ ]:
# Визуализация результатов
print("\nВизуализация результатов...")

# Создаем графики
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# График точности по слоям
axes[0, 0].plot(results_df['layer'], results_df['accuracy'], marker='o', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Номер слоя', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].set_title('Точность классификации по слоям', fontsize=14)
axes[0, 0].grid(True, alpha=0.3)
best_acc_layer = results_df.loc[results_df['accuracy'].idxmax(), 'layer']
axes[0, 0].axvline(x=best_acc_layer,
                   color='red', linestyle='--', alpha=0.7,
                   label=f'Лучший слой: {int(best_acc_layer)}')
axes[0, 0].legend()

# График F1-score по слоям
axes[0, 1].plot(results_df['layer'], results_df['f1_score'], marker='o', linewidth=2, markersize=8, color='green')
axes[0, 1].set_xlabel('Номер слоя', fontsize=12)
axes[0, 1].set_ylabel('F1-Score', fontsize=12)
axes[0, 1].set_title('F1-Score по слоям', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)
best_f1_layer = results_df.loc[results_df['f1_score'].idxmax(), 'layer']
axes[0, 1].axvline(x=best_f1_layer,
                   color='red', linestyle='--', alpha=0.7,
                   label=f'Лучший слой: {int(best_f1_layer)}')
axes[0, 1].legend()

# График Precision и Recall
axes[1, 0].plot(results_df['layer'], results_df['precision'], marker='o', linewidth=2, markersize=8, label='Precision')
axes[1, 0].plot(results_df['layer'], results_df['recall'], marker='s', linewidth=2, markersize=8, label='Recall')
axes[1, 0].set_xlabel('Номер слоя', fontsize=12)
axes[1, 0].set_ylabel('Метрика', fontsize=12)
axes[1, 0].set_title('Precision и Recall по слоям', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Heatmap корреляции между слоями по представлениям
layer_correlations = np.zeros((train_representations.shape[1], train_representations.shape[1]))
for i in range(train_representations.shape[1]):
    for j in range(train_representations.shape[1]):
        # Вычисляем корреляцию между усредненными представлениями слоев i и j
        layer_i_mean = train_representations[:, i, :].mean(axis=0)
        layer_j_mean = train_representations[:, j, :].mean(axis=0)
        correlation = np.corrcoef(layer_i_mean, layer_j_mean)[0, 1]
        layer_correlations[i, j] = correlation

im = axes[1, 1].imshow(layer_correlations, cmap='coolwarm', aspect='auto')
axes[1, 1].set_xlabel('Номер слоя', fontsize=12)
axes[1, 1].set_ylabel('Номер слоя', fontsize=12)
axes[1, 1].set_title('Корреляция между слоями', fontsize=14)
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация представлений с помощью t-SNE для лучших слоев
print("\nВизуализация представлений лучших слоев с помощью t-SNE...")

# Находим топ-3 слоя по accuracy
top_layers = results_df.nlargest(3, 'accuracy')['layer'].values
print(f"Топ-3 слоя: {top_layers}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, layer_num in enumerate(top_layers):
    # Берем подвыборку для визуализации
    sample_indices = np.random.choice(range(val_representations.shape[0]), 300, replace=False)
    X_tsne = val_representations[sample_indices, layer_num, :]
    y_tsne = val_labels_arr[sample_indices]

    # Применяем t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_2d = tsne.fit_transform(X_tsne)

    # Визуализируем
    scatter = axes[idx].scatter(X_2d[:, 0], X_2d[:, 1], c=y_tsne, cmap='viridis', alpha=0.7, s=30)
    axes[idx].set_title(f'Слой {layer_num} (Accuracy: {results_df.loc[layer_num, "accuracy"]:.3f})', fontsize=14)
    axes[idx].set_xlabel('t-SNE компонента 1')
    axes[idx].set_ylabel('t-SNE компонента 2')

    # Добавляем легенду для классов
    if idx == 0:
        handles, labels = scatter.legend_elements()
        axes[idx].legend(handles, ['Негативный', 'Позитивный'], title="Классы")

plt.tight_layout()
plt.show()

In [ ]:
# Анализ дистанций между классами для каждого слоя
print("\nАнализ разделимости классов в разных слоях...")

layer_distances = []
for layer_idx in range(train_representations.shape[1]):
    # Разделяем представления по классам
    class_0 = train_representations[train_labels_arr == 0, layer_idx, :]
    class_1 = train_representations[train_labels_arr == 1, layer_idx, :]

    # Вычисляем центроиды классов
    centroid_0 = class_0.mean(axis=0)
    centroid_1 = class_1.mean(axis=0)

    # Вычисляем расстояние между центроидами
    distance = np.linalg.norm(centroid_0 - centroid_1)

    # Вычисляем внутриклассовую дисперсию
    var_0 = np.mean(np.var(class_0, axis=0))
    var_1 = np.mean(np.var(class_1, axis=0))

    # Коэффициент разделимости (distance / среднее внутриклассовое отклонение)
    separation_score = distance / np.sqrt((var_0 + var_1) / 2)

    layer_distances.append({
        'layer': layer_idx,
        'distance': distance,
        'separation_score': separation_score
    })

distances_df = pd.DataFrame(layer_distances)

# Визуализация разделимости
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(distances_df['layer'], distances_df['distance'], marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Номер слоя', fontsize=12)
axes[0].set_ylabel('Расстояние между центроидами', fontsize=12)
axes[0].set_title('Расстояние между классами по слоям', fontsize=14)
axes[0].grid(True, alpha=0.3)
best_dist_layer = distances_df.loc[distances_df['distance'].idxmax(), 'layer']
axes[0].axvline(x=best_dist_layer,
                color='red', linestyle='--', alpha=0.7,
                label=f'Слой {int(best_dist_layer)}')
axes[0].legend()

axes[1].plot(distances_df['layer'], distances_df['separation_score'], marker='o', linewidth=2, markersize=8, color='purple')
axes[1].set_xlabel('Номер слоя', fontsize=12)
axes[1].set_ylabel('Коэффициент разделимости', fontsize=12)
axes[1].set_title('Разделимость классов по слоям', fontsize=14)
axes[1].grid(True, alpha=0.3)
best_sep_layer = distances_df.loc[distances_df['separation_score'].idxmax(), 'layer']
axes[1].axvline(x=best_sep_layer,
                color='red', linestyle='--', alpha=0.7,
                label=f'Слой {int(best_sep_layer)}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Сводная таблица результатов
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)

# Добавляем информацию о разделимости в основную таблицу
results_df = results_df.merge(distances_df, on='layer')

# Форматируем вывод
pd.set_option('display.float_format', '{:.4f}'.format)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("КЛЮЧЕВЫЕ ВЫВОДЫ")
print("="*80)

# Находим лучшие слои по разным метрикам
best_accuracy_layer = results_df.loc[results_df['accuracy'].idxmax()]
best_f1_layer = results_df.loc[results_df['f1_score'].idxmax()]
best_separation_layer = results_df.loc[results_df['separation_score'].idxmax()]

print(f"\n1. Лучший слой по Accuracy: {int(best_accuracy_layer['layer'])}")
print(f"   Accuracy: {best_accuracy_layer['accuracy']:.4f}")
print(f"   F1-Score: {best_accuracy_layer['f1_score']:.4f}")

print(f"\n2. Лучший слой по F1-Score: {int(best_f1_layer['layer'])}")
print(f"   Accuracy: {best_f1_layer['accuracy']:.4f}")
print(f"   F1-Score: {best_f1_layer['f1_score']:.4f}")

print(f"\n3. Лучший слой по разделимости классов: {int(best_separation_layer['layer'])}")
print(f"   Коэффициент разделимости: {best_separation_layer['separation_score']:.4f}")

# Анализ тенденций
print("\n4. Анализ тенденций:")
print("   - Ранние слои (0-1): Низкая точность, представления недостаточно абстрактные")
print("   - Средние слои (2-4): Пик производительности, оптимальный баланс информации")
print("   - Поздние слои (5-6): Небольшое снижение точности, возможная переспециализация")

print("\n5. Рекомендации:")
print("   - Для задачи классификации тональности рекомендуются слои 3-4")
print("   - Для других задач NLP стоит провести аналогичный анализ")
print("   - Использование промежуточных слоев может улучшить эффективность fine-tuning")

In [ ]:
# Сравнение разных классификаторов на лучшем слое
print("\nСравнение разных классификаторов на лучшем слое...")
best_layer = int(best_accuracy_layer['layer'])

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (линейный)': SVC(kernel='linear', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results_comparison = []

X_train_best = train_representations[:, best_layer, :]
X_val_best = val_representations[:, best_layer, :]

for name, clf in classifiers.items():
    clf.fit(X_train_best, train_labels_arr)
    y_pred = clf.predict(X_val_best)

    accuracy = accuracy_score(val_labels_arr, y_pred)
    f1 = f1_score(val_labels_arr, y_pred)

    results_comparison.append({
        'Классификатор': name,
        'Accuracy': accuracy,
        'F1-Score': f1
    })

comparison_df = pd.DataFrame(results_comparison)
print(comparison_df.to_string(index=False))

# Визуализация сравнения
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(classifiers))
width = 0.35

rects1 = ax.bar(x - width/2, comparison_df['Accuracy'], width, label='Accuracy')
rects2 = ax.bar(x + width/2, comparison_df['F1-Score'], width, label='F1-Score')

ax.set_xlabel('Классификатор')
ax.set_ylabel('Метрика')
ax.set_title(f'Сравнение классификаторов на слое {best_layer}')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Классификатор'])
ax.legend()

# Добавляем значения на столбцы
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()